# 04 — One-stage anchor-based: RetinaNet

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from vod import CLASSES
from vod.data import find_data_root, find_input_file, load_split
from vod.protocol import PROTOCOL
from vod.tv import (RETINA_SIZES, FrameDataset, anchor_match_stats, build_model, evaluate_run,
                    seed_everything, train)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")

## Data (same protocol as every detector)

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
print(len(train_frames), "train frames,", len(val_frames), "val frames")

## Anchors grounded in the training boxes

In [ ]:
# aspect ratios (h/w) at the 1/6, 1/2 and 5/6 quantiles of the training boxes
wh = np.concatenate([train_targets[i][0][:, 2:] - train_targets[i][0][:, :2] for i in train_frames["id"]])
log_ratio = np.log(wh[:, 1] / wh[:, 0])
DATA_RATIOS = tuple(np.round(np.exp(np.quantile(log_ratio, [1 / 6, 1 / 2, 5 / 6])), 2).tolist())
print("data aspect ratios h/w:", DATA_RATIOS)

ANCHOR_CANDIDATES = {
    "default 32-512, ratios 0.5/1/2": (RETINA_SIZES, (0.5, 1.0, 2.0)),
    "16-256, ratios 0.5/1/2": ((16, 32, 64, 128, 256), (0.5, 1.0, 2.0)),
    f"16-256, ratios {DATA_RATIOS}": ((16, 32, 64, 128, 256), DATA_RATIOS),
    f"12-192, ratios {DATA_RATIOS}": ((12, 24, 48, 96, 192), DATA_RATIOS),
}
coverage = {}
for name, (sizes, ratios) in ANCHOR_CANDIDATES.items():
    for fg in [0.5, 0.4]:
        t = anchor_match_stats(train_frames, train_targets, sizes, ratios, PROTOCOL["imgsz"], fg)
        coverage[(name, fg)] = t
        print(f"\n{name}, fg IoU {fg}: {t.attrs['anchors_per_image']} anchors/image, "
              f"negatives per positive anchor ~{t.attrs['negative_to_positive']}")
        display(t)

In [ ]:
summary = pd.DataFrame({k: v["no_positive_anchor"] for k, v in coverage.items()}).T
summary.index.names = ["anchors", "fg_iou"]
summary.columns = [f"no positive, {c}" for c in summary.columns]
summary

## Experiments
Each run changes one thing relative to the previous one. Run one per Kaggle version.

In [ ]:
EXPERIMENTS = {
    "retina_default": dict(arch="retinanet", lr=0.0025),
    "retina_anchors": dict(arch="retinanet", lr=0.0025, anchor_sizes=(16, 32, 64, 128, 256),
                           aspect_ratios=DATA_RATIOS),
    "retina_anchors_iou": dict(arch="retinanet", lr=0.0025, anchor_sizes=(16, 32, 64, 128, 256),
                               aspect_ratios=DATA_RATIOS, fg_iou=0.4, bg_iou=0.3),
    "retina_anchors_iou_focal": dict(arch="retinanet", lr=0.0025, anchor_sizes=(16, 32, 64, 128, 256),
                                     aspect_ratios=DATA_RATIOS, fg_iou=0.4, bg_iou=0.3, focal_gamma=1.5),
}
EXPERIMENT = "retina_default"
WEIGHTS = None  # e.g. "best.pt" attached as input, to skip training

cfg = {**PROTOCOL, **EXPERIMENTS[EXPERIMENT]}
out_dir = WORK / "runs" / EXPERIMENT
out_dir.mkdir(parents=True, exist_ok=True)
cfg

## Train

In [ ]:
seed_everything(cfg["seed"])
model = build_model(cfg)
train_ds = FrameDataset(train_frames, train_targets, cfg, train=True)
val_epoch_ds = FrameDataset(val_frames.iloc[::cfg["val_step_epoch"]], val_targets, cfg)

if WEIGHTS:
    model.load_state_dict(torch.load(find_input_file(WEIGHTS), map_location="cpu"))
else:
    history = train(model, train_ds, val_epoch_ds, VAL_GT, cfg, out_dir)
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location="cpu"))
    display(history)

## Evaluation on every val frame

In [ ]:
metrics, val_dets = evaluate_run(model, FrameDataset(val_frames, val_targets, cfg), VAL_GT, cfg, out_dir, EXPERIMENT)
pd.Series({k: v for k, v in metrics.items() if k != "config"}).to_frame(EXPERIMENT)

## Comparison with every finished run
Attach earlier versions' outputs (Add Input → Notebook Output) to include their `metrics.json`,
including the tuned Faster R-CNN from 03.

In [ ]:
roots = [WORK, Path("/kaggle/input")]
runs = {}
for root in roots:
    for p in root.rglob("metrics.json") if root.exists() else []:
        m = json.loads(p.read_text())
        runs[m["name"]] = m
        hist = p.parent / "history.csv"
        if hist.exists():
            runs[m["name"]]["history"] = pd.read_csv(hist)

cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "ms_per_image"]
comparison = pd.DataFrame({n: {c: m.get(c) for c in cols} for n, m in runs.items()}).T.round(4)
display(comparison)
pd.DataFrame({n: {c: m.get(f"AP_{c}") for c in CLASSES} for n, m in runs.items()}).T.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if "history" in m:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (subset)", title="Validation mAP during training")
ax.legend()
plt.tight_layout()
plt.savefig(out_dir / "map_curves.png", dpi=110)